# Базовая генерация статей

### Переменные окружения

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Для деталей см .env
cur_dir = Path.cwd()
env_path = cur_dir.parent / '.env'
load_dotenv(dotenv_path=env_path)

True

### Работы

Подключимся к API

In [2]:
import json
from tqdm import tqdm

import tiktoken
from langchain_gigachat.chat_models import GigaChat
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
gigachat_credentials = os.getenv("AUTHORIZATION_KEY")

In [ ]:
gigachat = GigaChat(
    credentials=gigachat_credentials, 
    model="GigaChat",
    temperature=0,
    verify_ssl_certs=False
)

gemini = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.7,
)

In [5]:
response = gemini.invoke("Сколько будет 24543 * 94925")
print(response.content[0]['text'])

24 543 * 94 925 = **2 329 744 275**


Теперь прочитаем онтологию и выделим из неё список понятий

In [6]:
with open(Path.cwd().parent / 'ontology.json', 'r', encoding='utf-8') as file:
    ontology = json.load(file)
ontology.keys()

dict_keys(['labels', 'edges'])

In [7]:
concepts = [
    (id, concept)
    for id, concept in ontology['labels'].items()
    if id[0] == 'Q'
]
concepts[:10]

[('Q1', 'Вселенная'),
 ('Q102836', 'пружина'),
 ('Q104837', 'термодинамическая фаза'),
 ('Q1075', 'цвет'),
 ('Q107715', 'физическая величина'),
 ('Q11023', 'инженерное дело'),
 ('Q11223329', 'динамометр'),
 ('Q11344', 'химический элемент'),
 ('Q11369', 'молекула'),
 ('Q11376', 'ускорение')]

In [8]:
len(concepts)

168

Павел подобрал промпт, который генерирует более менее неплохие статьи:

In [9]:
system_prompt = """\
Ты — автор детской энциклопедии по физике для учащихся 8 класса. Твоя задача — писать понятные, научно точные, но при этом интересные и живые статьи.

Строго соблюдай структуру в формате GitHub Flavored Markdown:

# [Название термина или явления]

## Краткое определение
[Дай четкое научное определение явления 2-3 предложениями. Обязательно добавь простую формулировку "Простыми словами: ..." после определения.]

## Основная идея
[Объясни физическую суть явления подробно, но без сложных формул. Расскажи, как это работает с научной точки зрения, упомяни ключевых ученых, если это уместно, и важные закономерности. Используй сравнения и метафоры. 4-6 предложений.]

## Примеры из жизни
[Приведи 3-5 конкретных примеров, где это явление встречается в повседневной жизни, технике или природе. Каждый пример описывай кратко, но информативно — почему это работает именно так.]

## Интересный факт
[Один короткий, но удивительный или малоизвестный факт по теме, который вызовет реакцию "Ничего себе!"]

Важные требования:
- Пиши на русском языке
- Сохраняй научную достоверность, но объясняй доступно для 8-классников
- Избегай сложных математических формул
- Текст должен быть живым, с примерами и аналогиями
- Строго соблюдай указанную структуру и заголовки
- Используй форматирование GitHub Markdown (жирный текст, списки где уместно)
"""

Забьём промпт в Gigachat вручную (оконная версия, чтобы не тратить API квоту), проверим сколько токенов мы потратили на системный промпт и генерацию:

In [10]:
encoding = tiktoken.get_encoding("cl100k_base")

In [11]:
article = gemini.invoke(system_prompt + 'Электромагнитная индукция').content[0]['text']
len(encoding.encode(system_prompt + article))

1748

In [12]:
print(article)

# Электромагнитная индукция

## Краткое определение
Электромагнитная индукция — это физическое явление, при котором в замкнутом проводнике возникает электрический ток при изменении магнитного поля, пронизывающего этот проводник. Этот процесс позволяет превращать механическое движение в чистую электрическую энергию.

**Простыми словами:** это способ «добывать» электричество с помощью магнита. Если быстро двигать магнит рядом с проводом, в проводе сам собой появится ток.

## Основная идея
Долгое время ученые знали, что электричество создает магнетизм, но английский физик **Майкл Фарадей** в 1831 году доказал, что работает и обратный процесс. Представьте, что электроны в металлическом проводе — это сонные пассажиры в вагоне, а магнитное поле — это невидимая рука. Если магнит неподвижен, «рука» ничего не делает, и электроны стоят на месте. Но стоит начать быстро двигать магнит или вращать катушку с проводом, как магнитные линии начинают «задевать» электроны, подталкивая их и заставляя бежа

Итого 1838 токенов. Возьмём 2000 как оценка сверху. У нас в бесплатной квоте (на одного человека) 800 тысяч токенов.

In [13]:
print(f'Всего {800_000 // 2000} статей можно нагенерировать')
print(f'То есть на каждую статью у нас примерно {(800_000 // 2000) / len(concepts) : .2f} попыток сгенерировать хорошо')

Всего 400 статей можно нагенерировать
То есть на каждую статью у нас примерно  2.38 попыток сгенерировать хорошо


У нас есть как минимум 3 ключа, то есть в три раза больше попыток, а также другие сервисы. Для zero-shot генерации придётся использовать модели по умнее, но не факт, что результат получится хорошим. Для более простых целей можно использовать квоту на openrouter - там есть qwen2.5.

Теперь собственно говоря сгенерируем все статьи

In [17]:
gemini = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.7,
)

In [18]:
all_tokens = 0
cnt = 0

with tqdm(concepts, unit="concept") as pbar:
    for id, concept in pbar:
        # Формируем путь заранее для проверки
        output_path = Path.cwd().parent / 'papers' / f'{id}.md'
        
        # Если файл уже существует, пропускаем генерацию
        if output_path.exists():
            continue
            
        prompt = f'{system_prompt}. <CONCEPT>{concept}</CONCEPT>'
        response = gemini.invoke(prompt)
        article = response.content[0]['text']
        
        # Сохраняем файл
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, mode='w', encoding='utf-8') as file:
            file.write(article)
        
        # Считаем токены
        tokens = len(encoding.encode(prompt + article))
        all_tokens += tokens
        cnt += 1
        
        # Вычисляем среднее
        mean_tokens = all_tokens / cnt
        pbar.set_description(f"Mean Tokens: {mean_tokens:.2f}")

print(f"\nTotal tokens processed: {all_tokens}")

Mean Tokens: 1664.13: 100%|██████████| 168/168 [10:02<00:00,  3.59s/concept]


Total tokens processed: 261269
